# 🎯 [K13_M01] 교육 데이터 구조화하기
### Google Colab 실습 환경 (교육훈련·역량개발 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K13/k13_m01_student.ipynb)


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 교육 데이터 구조화하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **가상 교육 콘텐츠 40개와 장병 60명의 수강 이력·평점을 표로 구조화하고 형식을 검사합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **콘텐츠(주제·난이도·시간·설명), 장병(계급·관심 주제), 평점 표를 만들고 주제별 개수와 평점 분포, 비어 있는 칸의 비율(희소도)을 확인하고 유효성을 점검하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `데이터 표 3종, 형식 검사 함수와 시험 결과, 학습·평가 분리 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 데이터 구조화 — 추천에 쓸 데이터를 일관된 표로 정리하고 형식 오류를 잡아내는 검사와 평가용 분리를 만들었는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(데이터 표 3종, 형식 검사 함수와 시험 결과, 학습·평가 분리 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""가상 교육 콘텐츠 40개와 장병 60명의 수강 이력·평점을 표로 구조화하고, 형식 검사와 희소도(비어 있는 정도)를 확인한다."""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()


def check_ratings(df, course_ids, soldier_ids):
    """평점 표의 오류(없는 장병·콘텐츠, 범위 밖 평점, 중복 수강)를 찾아 문장으로 돌려준다."""
    # [TODO : 학생 실습 1] 교육 데이터 구조화하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
print("[콘텐츠 표 일부]")
print(courses[["id", "title", "topic", "level", "hours"]].head(8).to_string(index=False))
print("\n주제별 콘텐츠 수:", courses["topic"].value_counts().to_dict())
print("난이도별 콘텐츠 수:", {LEVELS[k]: v for k, v in courses["level"].value_counts().sort_index().items()})
print(f"평균 교육 시간 {courses['hours'].mean():.1f}시간")

print("\n[장병 표 일부]")
print(soldiers.head(5).to_string(index=False))
print("계급별 인원:", soldiers["rank"].value_counts().reindex(RANKS).to_dict())

print(f"\n수강 기록 {len(ratings)}건, 장병 1인당 평균 {len(ratings) / len(soldiers):.1f}개 수강")
print("평점 분포:", ratings["rating"].value_counts().sort_index().to_dict())
full = ratings.pivot(index="soldier", columns="course", values="rating")
sparsity = full.isna().mean().mean()
print(f"장병x콘텐츠 행렬 {full.shape}, 비어 있는 칸 {sparsity:.0%} (대부분의 콘텐츠를 아직 안 들음)")
print(f"가장 적게 수강된 콘텐츠: {ratings['course'].value_counts().idxmin()} ({ratings['course'].value_counts().min()}건)")

print("\n[형식 검사]")
print("정상 데이터 오류:", check_ratings(ratings, CID, SID))
bad = pd.DataFrame([("S99", "C01", 4), ("S01", "C99", 3), ("S02", "C02", 7), ("S03", "C03", 4), ("S03", "C03", 5)],
                   columns=["soldier", "course", "rating"])
print("오류 주입 시험:", check_ratings(bad, CID, SID))

print(f"\n[평가용 분리] 학습 {len(train)}건, 평가용(평점 4 이상 중 숨긴 기록) {len(test)}건, 평가 대상 장병 {len(relevant)}명")


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[K13_M01] 교육 데이터 구조화하기 실행 완료
주요 지표 정상 산출 및 PASS 판정 완료.
```
</details>
